# Cohen’s Kappa in Machine Learning

## Is Kappa a useful accuracy measure?

Yes — **Cohen’s Kappa (κ)** can be a very useful evaluation measure in machine learning, but it is most helpful in the **right situations**.

---

## What Kappa measures

Kappa measures **agreement beyond chance**.

- **Accuracy** asks: *How often was the model correct?*
- **Kappa** asks: *How often was the model correct after accounting for agreement that could happen just by chance?*

This makes Kappa especially useful when a dataset has **imbalanced classes**.

---

## When Kappa is useful

### 1. When classes are imbalanced
If one class appears much more often than another, plain accuracy can look high even when the model is not truly useful.

**Example:**
If 90% of observations are `"No Fraud"`, then a model that always predicts `"No Fraud"` gets **90% accuracy**.

That sounds good at first, but it is not actually detecting fraud.

Kappa helps reveal this by giving a much lower score when the model is mostly benefiting from the class imbalance.

### 2. When you want performance beyond random guessing
Kappa adjusts for the fact that some agreement happens by chance.

A higher Kappa means the model is doing something genuinely useful beyond simply matching the dominant class.

### 3. For classification problems
Kappa is used for **categorical outcomes**, not regression.

---

## When Kappa is less useful

### 1. When predicted probabilities matter
If you care about probability estimates, metrics such as:

- **ROC-AUC**
- **Log loss**

may be better choices.

### 2. In some extreme imbalance situations
Kappa can sometimes behave in ways that are less intuitive when one class is extremely rare.

### 3. When you need an easy metric to explain
Accuracy, precision, recall, and F1-score are often easier to explain to students, managers, or non-technical audiences.

---

## Rule-of-thumb interpretation of Kappa

| Kappa | Interpretation |
|---|---|
| < 0.00 | Worse than random |
| 0.00–0.20 | Slight agreement |
| 0.21–0.40 | Fair agreement |
| 0.41–0.60 | Moderate agreement |
| 0.61–0.80 | Substantial agreement |
| 0.81–1.00 | Almost perfect agreement |

These are rough guidelines, not strict rules.

---

## Simple intuition

A simple way to think about Kappa is:

> “Some correct answers would happen just by luck.  
> Kappa removes that luck and tells us how much real skill the model shows.”

---

## Bottom line

Kappa is a useful metric when:

- you are working on a **classification** problem,
- your classes are **imbalanced**, and
- you want something more informative than plain accuracy.

However, it should usually be used **alongside** other measures such as:

- accuracy,
- confusion matrix,
- precision,
- recall,
- F1-score.


## A quick Python example

In the example below, we create an imbalanced dataset where 90% of cases belong to one class.

Then we compare two models:

1. A **naive model** that predicts only the majority class
2. A **better model** that correctly finds some of the minority cases

This helps show why **accuracy alone can be misleading**, and why **Kappa can be more informative**.


In [1]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, classification_report
import pandas as pd

# True labels: 90 zeros and 10 ones
y_true = [0] * 90 + [1] * 10

# Model 1: predicts only the majority class
y_pred_naive = [0] * 100

# Model 2: does a better job identifying some minority cases
y_pred_better = [0] * 85 + [1] * 5 + [0] * 5 + [1] * 5

def summarize_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    print(f"=== {name} ===")
    print(f"Accuracy: {acc:.3f}")
    print(f"Cohen's Kappa: {kappa:.3f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=3))

summarize_model("Naive Majority-Class Model", y_true, y_pred_naive)
print("\n" + "-"*60 + "\n")
summarize_model("Better Model", y_true, y_pred_better)


=== Naive Majority-Class Model ===
Accuracy: 0.900
Cohen's Kappa: 0.000

Confusion Matrix:
[[90  0]
 [10  0]]

Classification Report:
              precision    recall  f1-score   support

           0      0.900     1.000     0.947        90
           1      0.000     0.000     0.000        10

    accuracy                          0.900       100
   macro avg      0.450     0.500     0.474       100
weighted avg      0.810     0.900     0.853       100


------------------------------------------------------------

=== Better Model ===
Accuracy: 0.900
Cohen's Kappa: 0.444

Confusion Matrix:
[[85  5]
 [ 5  5]]

Classification Report:
              precision    recall  f1-score   support

           0      0.944     0.944     0.944        90
           1      0.500     0.500     0.500        10

    accuracy                          0.900       100
   macro avg      0.722     0.722     0.722       100
weighted avg      0.900     0.900     0.900       100



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## What to notice

### Naive majority-class model
This model predicts only the most common class.

- Its **accuracy will look high**
- But its **Kappa will be low**, because it is not really learning much beyond the dominant class

### Better model
This model identifies at least some minority cases correctly.

- Accuracy may improve only a little
- But Kappa often improves more clearly, because the model is doing better than chance

This is why Kappa can be a helpful companion metric in imbalanced classification problems.


In [2]:
# Put the two models side by side in a small table
results = pd.DataFrame({
    "Model": ["Naive Majority-Class Model", "Better Model"],
    "Accuracy": [
        accuracy_score(y_true, y_pred_naive),
        accuracy_score(y_true, y_pred_better)
    ],
    "Cohen's Kappa": [
        cohen_kappa_score(y_true, y_pred_naive),
        cohen_kappa_score(y_true, y_pred_better)
    ]
})

results


,Model,Accuracy,Cohen's Kappa
0,Naive Majority-Class Model,0.9,0.000000
1,Better Model,0.9,0.444444


## Summary

Remember:

- **Accuracy** can look impressive when one class dominates
- **Kappa** checks whether that performance is actually meaningful
- **Use Kappa with other metrics**, not by itself


